# MAMA-MIA — 04: Ablatsiya uchun preprocessing variantlari

Maqola hissasi — qaysi preprocessing qadami markazlararo umumlashuvga qancha
hissa qo'shishini o'lchash. Buning uchun bir xil ma'lumotdan to'rt daraja
tayyorlanadi, har biri bitta qadam qo'shadi:

| Variant | Izotrop resampling | Fizik masshtab (resize emas) | Bitta ko'krak |
|---|---|---|---|
| **V0** sodda | — | — | — |
| **V1** | ✓ | — | — |
| **V2** | ✓ | ✓ | — |
| **V3** to'liq | ✓ | ✓ | ✓ |

Har bosqich bitta qadam qo'shadi, ya'ni V0→V1 farqi resampling'ning hissasi,
V1→V2 fizik masshtabniki, V2→V3 bitta ko'krakka keltirishniki.

V3 allaqachon tayyor (`mamamia-preprocess-v4` chiqishi). Bu notebook V0, V1, V2
ni yasaydi.

**V0 — sodda pipeline.** Ko'pchilik ish shunday boshlanadi: 2-o'qni kesim o'qi
deb hisoblash va tekislikda 192x192 ga resize qilish. Bizning birinchi
urinishimiz ham aynan shunday edi. I-SPY va NACT hajmlarida qalin o'q (2 mm)
— bu 0-o'q, 2-o'q emas; natijada V0 ularni noto'g'ri tekislikda kesadi va
60x256 tekislikni 192x192 ga, ya'ni bir o'q bo'ylab ~3 baravar cho'zadi.

**Kanonik yo'naltirish ablatsiyada yo'q — chunki u bu ma'lumotda hech narsa
o'zgartirmaydi.** MAMA-MIA-Lite nashr qiluvchisi hajmlarni allaqachon RAS+ ga
keltirgan (barcha holatlarda `raw_orient` = RAS). Kodda u xavfsizlik uchun
qoldirilgan, lekin o'lchanadigan qadam emas. Buni maqolada aytib o'tish kerak:
qayta yo'naltirish o'zi yetarli emas, kesim o'qi muammosini **resampling** hal
qiladi.

**Samaradorlik.** Har bir holat diskdan **bir marta** o'qiladi, uchala variant
o'sha massivdan hosil qilinadi. Kanonik yo'naltirish nibabel obyektini qayta
yuklamasdan, `apply_orientation` bilan bajariladi — bu `as_closest_canonical`
bilan bir xil natija beradi (24 ta o'q kombinatsiyasida tekshirilgan).

**Kaggle:** Accelerator = None (CPU), Internet = On. GPU kvotasi sarflanmaydi.
Taxminan 1.5-2.5 soat.

## 1. Sozlamalar

In [ ]:
!pip install -q huggingface_hub nibabel h5py

import os, re, gc, time, json, shutil, zipfile, tempfile, collections
import numpy as np
import nibabel as nib
import h5py
from nibabel.orientations import io_orientation, apply_orientation
from scipy.ndimage import zoom as ndzoom
from skimage.transform import resize as skresize
from skimage.filters import threshold_otsu
from huggingface_hub import hf_hub_download

REPO_ID = "YongchengYAO/MAMA-MIA-Lite"
PARTS   = ["data-part001.zip", "data-part002.zip"]

TARGET_SPACING = 1.5
OUT_SIZE       = 192
MARGIN         = 8
BILATERAL_MM   = 256      # 03-ishlovdagi bilan bir xil (taqsimotdan olingan)

VARIANTS = ["V0_naive", "V1_iso_resize", "V2_iso_scale"]

DRY_RUN = None            # son = sinov rejimi; None = hammasi

TMP = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else tempfile.gettempdir()
ZIPDIR  = os.path.join(TMP, "zips")
scratch = os.path.join(TMP, "scratch")
ROOT = "/kaggle/working/variants" if os.path.isdir("/kaggle/working") else "./variants"
for d in (ZIPDIR, scratch, ROOT):
    os.makedirs(d, exist_ok=True)
for v in VARIANTS:
    os.makedirs(os.path.join(ROOT, v), exist_ok=True)

print("Chiqish:", ROOT)
print("Variantlar:", ", ".join(VARIANTS))

## 2. Yuklash va indeks

In [ ]:
t0 = time.time()
zip_paths = []
for p in PARTS:
    print("yuklanmoqda:", p, flush=True)
    zip_paths.append(hf_hub_download(REPO_ID, p, repo_type="dataset", local_dir=ZIPDIR))
print(f"jami {time.time()-t0:.0f}s")

zips = {p: zipfile.ZipFile(p) for p in zip_paths}
index = {}
for p, z in zips.items():
    for n in z.namelist():
        if not n.endswith(".nii.gz"):
            continue
        cid = os.path.basename(n)[:-len(".nii.gz")]
        slot = "img" if n.startswith("Images/") else "msk" if n.startswith("Masks/") else None
        if slot:
            index.setdefault(cid, {})[slot] = (p, n)

complete = {c: v for c, v in index.items() if "img" in v and "msk" in v}
def source_of(cid): return cid.split("_")[0].upper()
print("To'liq juftliklar:", len(complete))
print(dict(collections.Counter(source_of(c) for c in complete)))

## 3. Funksiyalar

Bir marta o'qish, uchta variant. Kanonik yo'naltirish va spacing almashtirish
`apply_orientation` orqali — `as_closest_canonical` bilan aynan bir xil.

In [ ]:
def load_once(zpath, member, scratch):
    """Hajmni bir marta o'qiydi va XOM massiv + affine + spacing qaytaradi."""
    dst = os.path.join(scratch, "t_" + os.path.basename(member))
    with zips[zpath].open(member) as src, open(dst, "wb") as out:
        shutil.copyfileobj(src, out, length=1 << 20)
    obj = nib.load(dst)
    arr = np.asanyarray(obj.dataobj)
    aff = obj.affine.copy()
    zooms = tuple(float(z) for z in obj.header.get_zooms()[:3])
    axc = "".join(nib.aff2axcodes(aff))
    del obj
    os.remove(dst)
    return arr, aff, zooms, axc


def to_canonical(arr, aff, zooms):
    """RAS kanonik yo'nalish, hajmni qayta o'qimasdan."""
    ornt = io_orientation(aff)
    out = apply_orientation(arr, ornt)
    oz = [0.0] * 3
    for i in range(3):
        oz[int(ornt[i, 0])] = zooms[i]
    return out, tuple(oz)


def crop_z(img, msk, margin=MARGIN):
    zs = np.where(msk.any(axis=(0, 1)))[0]
    if len(zs) == 0:
        return None, None, None
    z0 = max(0, int(zs[0]) - margin)
    z1 = min(img.shape[2], int(zs[-1]) + 1 + margin)
    return img[:, :, z0:z1], msk[:, :, z0:z1], (z0, z1, len(zs))


def resize_plane(img, msk, size=OUT_SIZE):
    """Tekislikda resize — fizik masshtab SAQLANMAYDI (V0 va V1 uchun)."""
    nz = img.shape[2]
    i = skresize(img.astype(np.float32), (size, size, nz), order=1,
                 preserve_range=True, anti_aliasing=True)
    m = skresize(msk.astype(np.float32), (size, size, nz), order=0,
                 preserve_range=True, anti_aliasing=False)
    return i, (m > 0.5).astype(np.uint8)


def resample_iso(img, msk, zooms, target=TARGET_SPACING):
    f = tuple(z / target for z in zooms)
    if all(abs(x - 1.0) < 1e-3 for x in f):
        return img.astype(np.float32), msk.astype(np.uint8)
    i = ndzoom(img.astype(np.float32), f, order=1, mode="nearest")
    m = ndzoom(msk.astype(np.uint8), f, order=0, mode="nearest")
    return i, (m > 0).astype(np.uint8)


def foreground_bbox(vol):
    sub = vol[::2, ::2, ::2].astype(np.float32)
    try:
        thr = float(threshold_otsu(sub))
    except Exception:
        thr = float(np.percentile(sub, 60))
    fg = vol > thr
    if not fg.any():
        thr = float(np.percentile(vol, 60)); fg = vol > thr
        if not fg.any():
            return (0, vol.shape[0], 0, vol.shape[1]), 0.0
    xs = np.where(fg.any(axis=(1, 2)))[0]
    ys = np.where(fg.any(axis=(0, 2)))[0]
    return (int(xs[0]), int(xs[-1]) + 1, int(ys[0]), int(ys[-1]) + 1), thr


def window_fit(vol, size, center_row, pad_value=0.0):
    """Fizik masshtabni SAQLAYDI — resize emas, kesish/to'ldirish."""
    h, w = vol.shape[0], vol.shape[1]
    out = np.full((size, size, vol.shape[2]), pad_value, dtype=vol.dtype)
    sy0 = int(round(center_row - size / 2)); sy0 = max(0, min(sy0, max(0, h - size)))
    sy1 = min(h, sy0 + size)
    sx0 = max(0, (w - size) // 2); sx1 = min(w, sx0 + size)
    dy0 = max(0, (size - (sy1 - sy0)) // 2); dy1 = dy0 + (sy1 - sy0)
    dx0 = max(0, (size - (sx1 - sx0)) // 2); dx1 = dx0 + (sx1 - sx0)
    out[dy0:dy1, dx0:dx1, :] = vol[sy0:sy1, sx0:sx1, :]
    return out


def to_uint16(vol, lo, hi):
    """Asl diapazonga qarab uint16. Intensivlik barcha variantlarda bir xil
    ishlanadi — ablatsiya faqat geometriyani o'zgartiradi."""
    if lo >= 0 and hi <= 65535:
        return np.clip(np.rint(vol), 0, 65535).astype(np.uint16)
    s = 65535.0 / (hi - lo) if hi > lo else 1.0
    return np.clip(np.rint((vol - lo) * s), 0, 65535).astype(np.uint16)


print("funksiyalar tayyor")

## 4. Variant qurish

Har bir holat uchun uchala variant bitta o'qishdan hosil qilinadi.

In [ ]:
def build_variants(cid, scratch):
    """Qaytadi: ({variant: [(key, img_u16, msk_u8, meta)]}, skip)"""
    zp_i, n_i = complete[cid]["img"]
    zp_m, n_m = complete[cid]["msk"]
    raw_img, aff, zooms, axc = load_once(zp_i, n_i, scratch)
    raw_msk, _, _, _          = load_once(zp_m, n_m, scratch)

    base = {"case": cid, "source": source_of(cid),
            "raw_shape": "x".join(map(str, raw_img.shape)),
            "raw_spacing": ",".join(f"{z:.3f}" for z in zooms),
            "raw_orient": axc,
            "slice_axis_raw": int(np.argmax(zooms))}

    if raw_img.shape != raw_msk.shape or not (raw_msk > 0).any():
        return None, {**base, "skip": "bad_pair"}

    lo, hi = float(raw_img.min()), float(raw_img.max())
    p99 = float(np.percentile(raw_img, 99))
    out = {v: [] for v in VARIANTS}

    # kanonik yo'nalish — bu ma'lumotda amalda no-op (hamma RAS),
    # lekin xavfsizlik uchun qo'llanadi va barcha variantlar uchun umumiy
    img, czooms = to_canonical(raw_img, aff, zooms)
    msk, _      = to_canonical((raw_msk > 0).astype(np.uint8), aff, zooms)
    del raw_img, raw_msk

    def emit(v, im, mk, extra):
        if mk.sum() == 0:
            return
        out[v].append((cid, to_uint16(im, lo, hi), mk,
            {**base, "variant": v, "key": cid, "side": "W",
             "n_slices": int(im.shape[2]), "tumor_vox": int(mk.sum()),
             "int_p99": p99, "int_min": lo, "int_max": hi, **extra}))

    # ---------- V0: resampling YO'Q, tekislikda resize ----------
    ic, mc, zinfo = crop_z(img, msk)
    if ic is not None:
        i0, m0 = resize_plane(ic, mc)
        emit("V0_naive", i0, m0, {"tumor_slices": zinfo[2], "bilateral": False})
        del i0, m0
    del ic, mc

    # ---------- izotrop (V1 va V2 uchun umumiy) ----------
    iso_i, iso_m = resample_iso(img, msk, czooms)
    del img, msk
    if not iso_m.any():
        return out, None

    bbox, _ = foreground_bbox(iso_i)
    lr_mm = (bbox[1] - bbox[0]) * TARGET_SPACING
    ic, mc, zinfo = crop_z(iso_i, iso_m)
    del iso_i, iso_m
    if ic is None:
        return out, None

    # ---------- V1: izotrop + resize (fizik masshtab yo'qoladi) ----------
    i1, m1 = resize_plane(ic, mc)
    emit("V1_iso_resize", i1, m1,
         {"tumor_slices": zinfo[2], "bilateral": bool(lr_mm > BILATERAL_MM),
          "fg_lr_mm": round(lr_mm, 1)})
    del i1, m1

    # ---------- V2: izotrop + fizik masshtab (kesish/to'ldirish) ----------
    pb, _ = foreground_bbox(ic)
    cr = (pb[0] + pb[1]) / 2.0
    i2 = window_fit(ic, OUT_SIZE, cr, pad_value=lo)
    m2 = window_fit(mc, OUT_SIZE, cr, pad_value=0).astype(np.uint8)
    emit("V2_iso_scale", i2, m2,
         {"tumor_slices": zinfo[2], "bilateral": bool(lr_mm > BILATERAL_MM),
          "fg_lr_mm": round(lr_mm, 1)})
    del ic, mc, i2, m2
    return out, None


# sinov: har manbadan bittadan
by_src = collections.defaultdict(list)
for c in sorted(complete):
    by_src[source_of(c)].append(c)

print("Sinov:")
for s in sorted(by_src):
    cid = by_src[s][0]
    t = time.time()
    res, skip = build_variants(cid, scratch)
    if skip:
        print(f"  {cid}: {skip['skip']}"); continue
    parts = "  ".join(f"{v.split('_')[0]}={res[v][0][1].shape if res[v] else 'yoq'}"
                      for v in VARIANTS)
    print(f"  {cid:<14} {parts}   {time.time()-t:.1f}s")

### Vizual taqqoslash

Bir xil holat uch variantda. I-SPY va NACT qatorlarida V0 cho'zilgan va
noto'g'ri tekislikda bo'lishi kerak, V1 esa to'g'ri — resampling'ning
ta'siri aynan shu yerda ko'rinadi.

In [ ]:
import matplotlib.pyplot as plt

show_cases = [by_src[s][0] for s in sorted(by_src)]
fig, axes = plt.subplots(len(show_cases), len(VARIANTS),
                         figsize=(3.6 * len(VARIANTS), 3.8 * len(show_cases)),
                         squeeze=False)

for r, cid in enumerate(show_cases):
    res, skip = build_variants(cid, scratch)
    for c, v in enumerate(VARIANTS):
        ax = axes[r][c]
        if skip or not res[v]:
            ax.text(.5, .5, "yo'q", ha="center"); ax.axis("off"); continue
        _, im, mk, meta = res[v][0]
        z = int(np.argmax(mk.sum(axis=(0, 1))))
        sl = im[:, :, z].astype(np.float32)
        a, b = np.percentile(sl, [1, 99.5])
        ax.imshow(np.rot90(np.clip(sl, a, b)), cmap="gray")
        ax.contour(np.rot90(mk[:, :, z]), levels=[.5], colors="red", linewidths=1.1)
        ax.set_title(f"{meta['source']} · {v.split('_')[0]}\nxom o'q={meta['slice_axis_raw']}",
                     fontsize=9)
        ax.axis("off")

plt.tight_layout(); plt.show()

## 5. To'liq qurish

In [ ]:
targets = sorted(complete) if DRY_RUN is None else sorted(complete)[:DRY_RUN]

handles = {v: {} for v in VARIANTS}      # variant -> source -> h5
metas   = {v: [] for v in VARIANTS}
skipped = []
t0 = time.time()

try:
    for i, cid in enumerate(targets, 1):
        res, skip = build_variants(cid, scratch)
        if skip:
            skipped.append(skip); continue
        for v in VARIANTS:
            for key, im, mk, meta in res[v]:
                src = meta["source"]
                if src not in handles[v]:
                    handles[v][src] = h5py.File(
                        os.path.join(ROOT, v, f"mamamia_{src}.h5"), "w")
                g = handles[v][src].create_group(key)
                g.create_dataset("img", data=im, compression="lzf",
                                 chunks=(OUT_SIZE, OUT_SIZE, 1))
                g.create_dataset("msk", data=mk, compression="lzf",
                                 chunks=(OUT_SIZE, OUT_SIZE, 1))
                metas[v].append(meta)
        del res

        if i % 25 == 0 or i == len(targets):
            el = time.time() - t0
            used = sum(os.path.getsize(os.path.join(dp, f))
                       for dp, _, fs in os.walk(ROOT) for f in fs) / 2**30
            print(f"{i:>5}/{len(targets)}  {el/60:5.1f} daq  "
                  f"qolgan ~{el/i*(len(targets)-i)/60:5.1f} daq  "
                  f"disk {used:4.1f} GB  o'tkazilgan {len(skipped)}", flush=True)
            gc.collect()
            if used > 16:
                print("[!] Disk chegarasi — to'xtatildi."); break
finally:
    for v in handles:
        for h in handles[v].values():
            h.close()

print(f"\nTugadi: {(time.time()-t0)/60:.1f} daqiqa, o'tkazilgan {len(skipped)}")

## 6. Xulosa

In [ ]:
import pandas as pd

for v in VARIANTS:
    df = pd.DataFrame(metas[v])
    df.to_csv(os.path.join(ROOT, v, "metadata.csv"), index=False)
    print(f"\n=== {v} ===  {len(df)} namuna")
    print(df.groupby("source").agg(
        namuna=("key", "count"),
        kesim=("n_slices", "mean"),
        osma=("tumor_vox", "mean"),
    ).round(1).to_string())

print("\nFayl hajmlari:")
tot = 0
for v in VARIANTS:
    sub = sum(os.path.getsize(os.path.join(ROOT, v, f))
              for f in os.listdir(os.path.join(ROOT, v))) / 2**20
    tot += sub
    print(f"  {v:<16} {sub:>8.1f} MB")
print(f"  {'JAMI':<16} {tot/1024:>8.2f} GB")

with open(os.path.join(ROOT, "variants_config.json"), "w") as f:
    json.dump({"variants": VARIANTS, "out_size": OUT_SIZE,
               "target_spacing": TARGET_SPACING, "margin": MARGIN,
               "bilateral_mm": BILATERAL_MM,
               "note": "V3 = mamamia-preprocess-v4 chiqishi (izotrop + fizik masshtab + bitta ko'krak)",
               "date": time.strftime("%Y-%m-%d")}, f, indent=2)
print("\nconfig yozildi")

## 7. Keyingi qadam

**Save Version → Save & Run All (Commit).** Tugagach bu notebook chiqishi
ablatsiya notebook'iga (`05_ablation`) kirish sifatida ulanadi — V3 esa
`mamamia-preprocess-v4` dan keladi.

Ablatsiya ikki foldda bajariladi: chetda **NACT** (intensivlik bo'yicha eng
uzoq, baseline'da eng past tashqi Dice 0.496) va chetda **DUKE** (eng yaqin,
0.620). Ikkala qutb ham qamrab olinadi.